# Whisper 逐字稿產生器

輸入 **YouTube URL** 或 **本機 MP3 檔案**，使用本機 Whisper 產出帶時間戳記的逐字稿。

支援在 **Google Colab** 或**本機 Jupyter** 執行。

### 前置需求
| 環境 | ffmpeg | Python 套件 |
|------|--------|-------------|
| Google Colab | 預裝，自動確認 | 下方 cell 自動安裝 |
| 本機 | 需手動安裝：Mac `brew install ffmpeg`/ Linux `apt install ffmpeg` | 下方 cell 自動安裝 |

In [1]:
# @title
%pip install openai-whisper yt-dlp --quiet

# 偵測執行環境
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab 上確保 ffmpeg 存在（通常預裝，這裡做保險）
if IN_COLAB:
    import subprocess
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True)
    if result.returncode != 0:
        subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], check=True)
        print("ffmpeg 安裝完成。")
    else:
        print("ffmpeg 已就緒（Colab 預裝）。")
else:
    import shutil
    if shutil.which("ffmpeg"):
        print("ffmpeg 已就緒（本機）。")
    else:
        print("⚠️  找不到 ffmpeg，請先安裝：brew install ffmpeg（Mac）或 apt install ffmpeg（Linux）")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 51.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.0 MB/s eta 0:00:00
ffmpeg 已就緒（Colab 預裝）。


In [2]:
# @title
import whisper
import yt_dlp
import os
import tempfile
from pathlib import Path
from datetime import datetime

## 設定

In [3]:
#@title ⚙️ 設定
WHISPER_MODEL = "medium" #@param ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3"]
LANGUAGE = "" #@param {type:"string"}
#@markdown > 語言留空 = 自動偵測；可填 `zh`、`en`、`ja` 等加速並提高準確度
MAX_CHARS_PER_LINE = 0 #@param {type:"integer"}
OUTPUT_FILE = "transcript.txt" #@param {type:"string"}
OUTPUT_SRT_FILE = "transcript.srt" #@param {type:"string"}

## 工具函式

In [4]:
# @title
def format_timestamp(seconds: float) -> str:
    """將秒數轉為 YouTube 風格時間戳記，例如 1:23 或 1:02:34"""
    total = int(seconds)
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


def format_srt_timestamp(seconds: float) -> str:
    """將秒數轉為 SRT 格式時間戳記，例如 00:00:01,000"""
    ms = round((seconds % 1) * 1000)
    total = int(seconds)
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def download_youtube_audio(url: str, output_dir: str) -> tuple[str, str]:
    """從 YouTube 下載音訊，回傳 (mp3路徑, 影片標題)"""
    ydl_opts = {
        "format": "bestaudio/best",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "outtmpl": os.path.join(output_dir, "%(id)s.%(ext)s"),
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        video_id = info["id"]
        title = info.get("title", video_id)
        mp3_path = os.path.join(output_dir, f"{video_id}.mp3")
        return mp3_path, title


def transcribe_audio(audio_path: str, model, language=None) -> list[dict]:
    """使用 Whisper 轉錄音訊，回傳片段清單"""
    opts = {"verbose": False}
    if language:
        opts["language"] = language
    result = model.transcribe(audio_path, **opts)
    return result["segments"]


def segments_to_transcript(segments: list[dict], max_chars: int = 0) -> str:
    """將 Whisper 片段轉為帶時間戳記的逐字稿文字"""
    lines = []
    for seg in segments:
        ts = format_timestamp(seg["start"])
        text = seg["text"].strip()
        if max_chars > 0 and len(text) > max_chars:
            chunks = [text[i:i+max_chars] for i in range(0, len(text), max_chars)]
            lines.append(f"[{ts}] {chunks[0]}")
            for chunk in chunks[1:]:
                lines.append(f"       {chunk}")
        else:
            lines.append(f"[{ts}] {text}")
    return "\n".join(lines)


def segments_to_srt(segments: list[dict]) -> str:
    """將 Whisper 片段轉為標準 SRT 字幕格式"""
    blocks = []
    for i, seg in enumerate(segments, start=1):
        start = format_srt_timestamp(seg["start"])
        end = format_srt_timestamp(seg["end"])
        text = seg["text"].strip()
        blocks.append(f"{i}\n{start} --> {end}\n{text}")
    return "\n\n".join(blocks)

## 輸入：選擇 YouTube URL 或 MP3 檔案

三選一填入，**同時填寫時優先順序：上傳 > YouTube URL > 本機路徑**

In [5]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=pTvSMjsYnok" #@param {type:"string"}
MP3_PATH = "" #@param {type:"string"}
UPLOAD_IN_COLAB = False #@param {type:"boolean"}

## 執行轉錄

In [6]:
# @title
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"運算裝置：{device.upper()}")

if device == "cpu" and IN_COLAB:
    print("\n💡 提示：可免費啟用 GPU 加速 Whisper（快約 5-10 倍）")
    print("   Colab 選單 → 執行階段 → 變更執行階段類型 → T4 GPU → 儲存")
    print("   重新連線後再執行此 notebook。\n")

print(f"載入 Whisper 模型：{WHISPER_MODEL} ...")
model = whisper.load_model(WHISPER_MODEL, device=device)
print("模型載入完成。")

運算裝置：CUDA
載入 Whisper 模型：medium ...


100%|██████████████████████████████████████| 1.42G/1.42G [00:11<00:00, 137MiB/s]


模型載入完成。


In [7]:
# @title
title = ""
_tmp_dir = None
_language = LANGUAGE.strip() or None  # 空字串轉 None

if UPLOAD_IN_COLAB:
    # ── 選項 C：Colab 上傳介面 ────────────────────────────────────
    if not IN_COLAB:
        raise RuntimeError("UPLOAD_IN_COLAB 僅支援 Google Colab 環境。")
    from google.colab import files
    print("請選擇要上傳的 MP3 檔案...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("未選擇任何檔案。")
    filename = list(uploaded.keys())[0]
    audio_path = f"/content/{filename}"
    title = Path(audio_path).stem
    print(f"已上傳：{audio_path}")

elif YOUTUBE_URL.strip():
    # ── 選項 A：YouTube ───────────────────────────────────────────
    import tempfile
    _tmp_dir = tempfile.mkdtemp()
    print(f"下載音訊中：{YOUTUBE_URL}")
    audio_path, title = download_youtube_audio(YOUTUBE_URL, _tmp_dir)
    print(f"下載完成：{title}")

elif MP3_PATH.strip():
    # ── 選項 B：本機 MP3 ──────────────────────────────────────────
    audio_path = MP3_PATH.strip()
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"找不到檔案：{audio_path}")
    title = Path(audio_path).stem
    print(f"使用本機檔案：{audio_path}")

else:
    raise ValueError("請填入 YOUTUBE_URL、MP3_PATH 其中一項，或勾選 UPLOAD_IN_COLAB。")

print(f"\n開始轉錄，模型：{WHISPER_MODEL}，語言：{_language or '自動偵測'} ...")
segments = transcribe_audio(audio_path, model, language=_language)
transcript = segments_to_transcript(segments, max_chars=MAX_CHARS_PER_LINE)

if _tmp_dir:
    import shutil
    shutil.rmtree(_tmp_dir, ignore_errors=True)

print("\n轉錄完成！")

下載音訊中：https://www.youtube.com/watch?v=pTvSMjsYnok
下載完成：史上最強智囊團：一個護航、一個挑刺，真相竟然越辯越明？Dual-AI Arena 幫你終結猶豫不決！

開始轉錄，模型：medium，語言：自動偵測 ...
Detected language: Chinese


100%|██████████| 61740/61740 [01:39<00:00, 621.24frames/s]


轉錄完成！


## 結果

In [8]:
# @title
header = f"# {title}\n# 產生時間：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
full_output = header + "\n" + transcript

print(full_output)

# 史上最強智囊團：一個護航、一個挑刺，真相竟然越辯越明？Dual-AI Arena 幫你終結猶豫不決！
# 產生時間：2026-05-05 12:14:24

[0:00] 大家好,我現在跟大家介紹我自己開發的一個小程式
[0:07] 小應用,它就是雙角色AI辯論擂台
[0:12] 那這個專案本來應該是我在學校的人工應用開發的一個課程裡面
[0:23] 要讓學生進行Five Coding,要使用的一個專案的範例
[0:28] 那我們現在先跟大家介紹一下,讓同學們也知道說
[0:33] 他要完成那個專案,他會長什麼樣子
[0:36] 好,那什麼是這個AI的雙角色的AI辯論的數位辯論擂台呢
[0:44] 這主要是因為我記得我們小時候有個童謠
[0:48] 就是有個阿公跟阿嬤,他們不是挖到一隻泥鰍
[0:52] 然後針對這個泥鰍呢,兩個人對於說泥鰍要煮鹹一點還是煮淡一點
[0:58] 兩個意見相容不下吵架,嬰兒還打破了鍋子
[1:02] 這是非常不划算的,對不對
[1:05] 所以現在我們有了AI之後,我們如果能夠透過AI來幫我們進行辯論
[1:10] 不要讓兩個人去吵架,可能就可以避免這種情況發生
[1:15] 到底是哪一種是應該是比較好的
[1:17] 所以我們就透過我們現在的AI這樣子來分別扮演兩個不同的角色
[1:22] 就好像周伯通一樣
[1:25] 他有一個左右互搏的方式,就左手跟右手打架的一個方式
[1:30] 那我們就讓AI分別扮演不同的角色來進行辯論
[1:36] 根據自己的立場提出自己的看法
[1:38] 那我們這套系統在使用的時候呢
[1:41] 唯一要準備的就是我們到使用Google的API Key
[1:45] 所以在使用之前呢,可能就是各位要先到Google的AI Studio
[1:50] 去找到這個API Key
[1:52] 然後再來使用
[1:54] 但是這個API Key只會存在你自己的本期端
[1:57] 不會上傳單一段,所以都不用擔心
[2:00] 那我們這一套服務的原理基本上就是
[2:06] 我們會有一個支持方跟一個反對方
[2:09] 然後我們會針對這兩個給他設定我們的系統提示詞
[2:14] 分別扮演不同的角色
[2:16] 可是因為不同的議題這個角色在扮演的時候
[2:19] 這個系統的提示詞可能會不一樣
[2:21]

In [9]:
# @title
if OUTPUT_FILE:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(full_output)
    print(f"TXT 已儲存至：{os.path.abspath(OUTPUT_FILE)}")

if OUTPUT_SRT_FILE:
    srt_content = segments_to_srt(segments)
    with open(OUTPUT_SRT_FILE, "w", encoding="utf-8") as f:
        f.write(srt_content)
    print(f"SRT 已儲存至：{os.path.abspath(OUTPUT_SRT_FILE)}")

if not OUTPUT_FILE and not OUTPUT_SRT_FILE:
    print("OUTPUT_FILE 與 OUTPUT_SRT_FILE 皆未設定，跳過儲存。")

TXT 已儲存至：/content/transcript.txt
SRT 已儲存至：/content/transcript.srt
